# 06.1 — Bahdanau Attention

**Problem it solves:** Seq2Seq compresses all encoder info into one vector. With attention, the decoder dynamically attends to different parts of the encoder output at each decoding step.

**Key idea:** At each decoder step t, compute an attention weight α_{t,i} for each encoder hidden state h_i. The context vector is the weighted sum: c_t = Σ α_{t,i} × h_i.

**The alignment matrix** (attention weights across all steps) visualizes what the decoder 'looks at' when generating each output token.

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class BahdanauAttention(nn.Module):
    """
    Additive attention (Bahdanau et al., 2015).
    
    score(h_i, s_t) = v^T tanh(W_1 h_i + W_2 s_t)
    
    where h_i = encoder hidden state at position i
          s_t = decoder hidden state at step t
    
    Also called 'additive attention' because it adds the two projections.
    """
    
    def __init__(self, encoder_dim, decoder_dim, attention_dim):
        super().__init__()
        self.W1 = nn.Linear(encoder_dim, attention_dim, bias=False)
        self.W2 = nn.Linear(decoder_dim, attention_dim, bias=False)
        self.v  = nn.Linear(attention_dim, 1, bias=False)
    
    def forward(self, encoder_outputs, decoder_hidden):
        """
        encoder_outputs: [batch, src_len, encoder_dim]
        decoder_hidden:  [batch, decoder_dim]
        Returns:
          context:       [batch, encoder_dim]
          attn_weights:  [batch, src_len]
        """
        # Project encoder outputs: [batch, src_len, attn_dim]
        energy_enc = self.W1(encoder_outputs)
        
        # Project decoder hidden: [batch, 1, attn_dim]
        energy_dec = self.W2(decoder_hidden).unsqueeze(1)
        
        # Compute attention energy scores
        # Broadcasts over src_len: [batch, src_len, attn_dim]
        energy = torch.tanh(energy_enc + energy_dec)
        
        # Scalar score per position: [batch, src_len]
        scores = self.v(energy).squeeze(-1)
        
        # Softmax to get attention weights
        attn_weights = F.softmax(scores, dim=1)
        
        # Context vector: weighted sum of encoder outputs
        # [batch, src_len] × [batch, src_len, enc_dim] -> [batch, enc_dim]
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs).squeeze(1)
        
        return context, attn_weights

# Test the attention mechanism
batch_size, src_len, enc_dim = 2, 8, 32
dec_dim, attn_dim = 32, 16

attn = BahdanauAttention(enc_dim, dec_dim, attn_dim)
enc_out = torch.randn(batch_size, src_len, enc_dim)
dec_hid = torch.randn(batch_size, dec_dim)

context, weights = attn(enc_out, dec_hid)
print(f"Encoder outputs: {enc_out.shape}")
print(f"Decoder hidden:  {dec_hid.shape}")
print(f"Context vector:  {context.shape}")
print(f"Attn weights:    {weights.shape}")
print(f"Weights sum to 1: {weights.sum(dim=1)}")  # should be all 1.0
print(f"Weights (batch 0): {weights[0].detach().numpy().round(3)}")

In [ ]:
# Visualize attention alignment
import matplotlib
matplotlib.use('Agg')  # non-interactive backend
import matplotlib.pyplot as plt

# Simulate attention for a translation task
# English -> (fake) target, showing alignment

src_words = ['The', 'cat', 'sat', 'on', 'the', 'mat', '.']
tgt_words = ['Le', 'chat', 'était', 'assis', 'sur', 'le', 'tapis', '.']

# Simulate ideal alignment weights
# Note: French 'chat' (cat) should attend to English 'cat', etc.
ideal_attention = np.array([
    [0.9, 0.0, 0.0, 0.0, 0.1, 0.0, 0.0],  # Le -> The
    [0.0, 0.9, 0.0, 0.0, 0.0, 0.1, 0.0],  # chat -> cat
    [0.1, 0.2, 0.5, 0.1, 0.1, 0.0, 0.0],  # était -> sat (approximate)
    [0.0, 0.0, 0.8, 0.1, 0.0, 0.0, 0.1],  # assis -> sat
    [0.0, 0.0, 0.0, 0.9, 0.1, 0.0, 0.0],  # sur -> on
    [0.0, 0.0, 0.0, 0.0, 0.9, 0.1, 0.0],  # le -> the
    [0.0, 0.1, 0.0, 0.1, 0.1, 0.6, 0.1],  # tapis -> mat
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0],  # . -> .
])

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(ideal_attention, cmap='Blues')
ax.set_xticks(range(len(src_words))); ax.set_xticklabels(src_words, fontsize=11)
ax.set_yticks(range(len(tgt_words))); ax.set_yticklabels(tgt_words, fontsize=11)
ax.set_xlabel('Source (English)', fontsize=12)
ax.set_ylabel('Target (French)', fontsize=12)
ax.set_title('Bahdanau Attention Alignment Matrix', fontsize=13)
plt.colorbar(im)
plt.tight_layout()
plt.savefig('attention_matrix.png', dpi=100, bbox_inches='tight')
plt.close()
print("Saved attention_matrix.png")
print()
print("Key observations:")
print("- Diagonal pattern: English and French are fairly aligned")
print("- Non-diagonal entries: where word order differs between languages")
print("- This visualization motivated much of NLP interpretability research")

In [ ]:
# Bahdanau vs Luong attention

class LuongAttention(nn.Module):
    """
    Multiplicative attention (Luong et al., 2015).
    score(h_i, s_t) = s_t^T W h_i  (or just s_t^T h_i for dot product)
    
    Faster than Bahdanau (no tanh), similar performance.
    """
    
    def __init__(self, hidden_dim, score_type='general'):
        super().__init__()
        self.score_type = score_type
        if score_type == 'general':
            self.W = nn.Linear(hidden_dim, hidden_dim, bias=False)
    
    def forward(self, encoder_outputs, decoder_hidden):
        # encoder_outputs: [batch, src_len, H]
        # decoder_hidden:  [batch, H]
        
        if self.score_type == 'dot':
            # Direct dot product
            scores = torch.bmm(encoder_outputs, decoder_hidden.unsqueeze(2)).squeeze(2)
        elif self.score_type == 'general':
            # s_t^T W h_i
            projected = self.W(encoder_outputs)  # [batch, src_len, H]
            scores = torch.bmm(projected, decoder_hidden.unsqueeze(2)).squeeze(2)
        
        attn_weights = F.softmax(scores, dim=1)
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs).squeeze(1)
        return context, attn_weights

print("Attention score functions:")
print()
print("Bahdanau (additive):   score = v^T tanh(W1*h_i + W2*s_t)")
print("  + Captures complex alignment")
print("  - Slower: tanh, extra projection")
print()
print("Luong dot-product:     score = s_t^T h_i")
print("  + Fastest: just a dot product")
print("  - Requires encoder/decoder to have same dimension")
print()
print("Luong general:         score = s_t^T W h_i")
print("  + Flexible dimensions, fast")
print()
print("Scaled dot-product:    score = (Q K^T) / sqrt(d_k)")
print("  This is what the Transformer uses. Next notebook.")

## Summary

**Bahdanau attention** solved the seq2seq bottleneck. Key ideas:
1. Each decoding step gets a fresh context vector tailored to what's needed
2. Attention weights are differentiable → learned end-to-end
3. Alignment matrix is interpretable

**Performance:** Attention-based seq2seq on WMT14 en-de: +3 BLEU over fixed context vector.

**What attention teaches:** The decoder doesn't need to memorize everything — it can look things up. This insight led directly to the full attention mechanism in Transformers.

---
**Next:** 06.2 — Scaled Dot-Product Attention (the Transformer's core operation)